In [1]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from sqlalchemy.orm import sessionmaker

params = quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=AI;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}",
    pool_pre_ping=True
)


SessionLocal = sessionmaker(
    bind=engine,
    autoflush=False,
    autocommit=False
)

In [2]:
from langchain_openai.chat_models import ChatOpenAI 
llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="not-needed",
    model="google/gemma-3-4b",
    temperature=0,
    max_completion_tokens=250
)

c:\Users\babaei.nima\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import pyodbc
from sqlalchemy import text

print(pyodbc.drivers())
with engine.connect() as conn:
    result = conn.execute(text("SELECT @@VERSION"))
    print(result.scalar())

['SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'SQL Server Native Client 11.0', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 17 for SQL Server']
Microsoft SQL Server 2019 (RTM) - 15.0.2000.5 (X64) 
	Sep 24 2019 13:48:23 
	Copyright (C) 2019 Microsoft Corporation
	Enterprise Edition: Core-based Licensing (64-bit) on Windows 10 Pro 10.0 <X64> (Build 26200: ) (Hypervisor)



In [4]:
from sqlalchemy import (
    Column,
    BigInteger,
    String,
    Text,
    DateTime
)

from sqlalchemy.orm import declarative_base

Base = declarative_base()


class ChatMessage(Base):
    __tablename__ = "ChatMessages"

    id = Column(BigInteger, primary_key=True)
    sessionid = Column(String(100))
    role = Column(String(20))
    content = Column(Text)
    createdat = Column(DateTime)


class ConversationSummary(Base):
    __tablename__ = "ConversationSummary"

    sessionid = Column(String(100), primary_key=True)
    summary = Column(Text)
    updatedat = Column(DateTime)

In [5]:
from sqlalchemy.orm import Session
from datetime import datetime, UTC

def save_message(
    db: Session,
    session_id: str,
    role: str,
    content: str
) -> None:
    msg = ChatMessage(
        sessionid=session_id,
        role=role,
        content=content,
        createdat = datetime.now(UTC)
    )

    db.add(msg)

In [16]:
db = SessionLocal()

save_message(db= db, 
             session_id="s1", 
             role="user", 
             content="what is the future of Production Planning?")
db.commit()
db.close()

In [18]:
from langchain_core.messages import (
    HumanMessage,
    AIMessage
)

rows = (
        db.query(ChatMessage)
        .filter(
            ChatMessage.sessionid == "s1"
        )
        .order_by(
            ChatMessage.createdat.desc()
        )
        .all()
    )

In [19]:
total_chars = 0
selected = []

for row in rows:

    content_length = len(row.content)

    if total_chars + content_length > 8000:
        break

    selected.append(row)
    total_chars += content_length

rows.reverse()

messages = []

for row in rows:

    if row.role == "user":
        messages.append(
            HumanMessage(
                content=row.content
            )
        )

    elif row.role == "system":
        messages.append(
            AIMessage(
                content=row.content
            )
        )

In [20]:
print(messages)
print(total_chars)

[HumanMessage(content='tell me about production planning', additional_kwargs={}, response_metadata={}), AIMessage(content="Production planning is the process of developing a strategy for the production of a company's products and services. It describes how goods will be manufactured, the expected demand for those goods, and any production requirements ", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='what is the future of Production Planning?', additional_kwargs={}, response_metadata={})]
306


In [11]:
summary_record = (
            db.query(ConversationSummary)
            .filter(
                ConversationSummary.sessionid == "s1"
            )
            .first()
        )

In [12]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)

In [13]:
rewrite_prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """
    Rewrite the user's latest question into a standalone question.

    Conversation Summary: {summary}

    Instructions:
        - Use the summary and chat history to understand references.
        - Preserve the original meaning.
        - Replace pronouns such as "it", "they", "this", "that" with their actual meaning when possible.
        - Return only the rewritten question.
    """
            ),

            MessagesPlaceholder(
                variable_name="chat_history"
            ),

            ("human", "{question}")
        ]
    )

In [14]:
rewrite_chain = (
        rewrite_prompt
        | llm
    )

In [21]:
rewrite_chain.invoke(
        {
            "chat_history": messages, #load_recent_history
            "question": "where should I start to learn it?", 
            "summary": summary_record #load_summary
        }
    ).content

'What are the key trends and predictions shaping the future of production planning? And where would you recommend starting to learn about this field?'

In [ ]:
summary_instructions = f'''
    summarize the latest user questions and system answers by by incorporating the new lines of conversation below.
    so that the result reflects the most recent context and developments.

    conversation messages: {messages}
    '''
#[HumanMessage(summary_instructions)]
summary = llm.invoke(summary_instructions)


In [26]:
print(summary.content)

Okay, here’s a summary incorporating the latest conversation:

The user initially asked about “production planning.” The AI responded with a definition of production planning as the strategy for manufacturing goods and services, including anticipated demand and production requirements.  Following this, the user then inquired about "what is the future of Production Planning?".



In [27]:
rows_s = (
        db.query(ChatMessage)
        .filter(
            ChatMessage.sessionid == "s1"
        )
        .order_by(
            ChatMessage.createdat.asc()
        )
        .limit(20)
        .all()
    )

print(rows_s)

[<__main__.ChatMessage object at 0x0000027C61032690>, <__main__.ChatMessage object at 0x0000027C6103DB50>, <__main__.ChatMessage object at 0x0000027C60C57E10>]


In [29]:
lines = []

for row in rows_s:

    lines.append(
        f"{row.role}: {row.content}"
    )

In [30]:
print("\n".join(lines))

user: tell me about production planning
system: Production planning is the process of developing a strategy for the production of a company's products and services. It describes how goods will be manufactured, the expected demand for those goods, and any production requirements 
user: what is the future of Production Planning?
